<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/Wechuli_Working_with_JSON_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================
# STEP 1: Load the JSON File
# ============================================
import json
import pandas as pd

# Upload the file in Colab first:
# from google.colab import files
# uploaded = files.upload()

# Load the JSON file
with open('sample.json', 'r') as f:
    data = json.load(f)

# Always inspect the structure first — this tells you what keys exist
print(type(data))
print(data)  # or print(data[0]) if it's a list, to see one record's shape

# ============================================
# TASK: Retrieve all order IDs with total amounts
# ============================================

# The JSON is a list of customer dictionaries, each containing an 'orders' key.
# We need to extract all 'orders' lists from each customer and flatten them into a single list.
all_orders = []
for customer_record in data:
    if 'orders' in customer_record:
        all_orders.extend(customer_record['orders'])

orders = all_orders

order_totals = [
    {'order_id': order['order_id'], 'total': order['total']}
    for order in orders
]

for item in order_totals:
    print(f"Order ID: {item['order_id']}, Total: {item['total']}")

# ============================================
# TASK: Normalize nested JSON into a flat DataFrame
# ============================================

# To correctly normalize, we need to start from the top-level 'data' list (customer records)
# and specify the path to 'items' through 'orders', while carefully pulling meta data.
df = pd.json_normalize(
    data,                                # Start from the top-level list of customer records
    record_path=['orders', 'items'],     # Path to the nested list of items within orders
    meta=[
        'customer_id',                   # customer_id is directly in the customer record
        ['orders', 'order_id'],          # order_id is in the 'orders' dict, one level up from 'items'
        ['orders', 'total']              # total is also in the 'orders' dict
    ]  # fields to carry over from parent records
)

# Rename columns generated by the meta path to match the requested format
df = df.rename(columns={'orders.order_id': 'order_id', 'orders.total': 'total'})

# Reorder columns to match the requested format, dropping 'price' as it's not in the desired output
df = df[['customer_id', 'order_id', 'product', 'quantity', 'total']]

print(df.head())
print(df.info())

# ============================================
# TASK: Save the processed DataFrame back into a new JSON file
# ============================================

df.to_json('orders_cleaned.json', orient='records', indent=2)

# Optional: download it directly
from google.colab import files
files.download('orders_cleaned.json')


<class 'list'>
[{'customer_id': 101, 'name': 'Alice Johnson', 'email': 'alice@example.com', 'orders': [{'order_id': 5001, 'date': '2024-02-01', 'total': 120.5, 'items': [{'product': 'Laptop', 'price': 1000.0, 'quantity': 1}, {'product': 'Mouse', 'price': 20.5, 'quantity': 1}], 'payment': {'method': 'Credit Card', 'status': 'Completed'}}]}, {'customer_id': 102, 'name': 'Bob Williams', 'email': 'bob@example.com', 'orders': [{'order_id': 5002, 'date': '2024-02-03', 'total': 250.0, 'items': [{'product': 'Smartphone', 'price': 700.0, 'quantity': 1}, {'product': 'Charger', 'price': 50.0, 'quantity': 2}], 'payment': {'method': 'PayPal', 'status': 'Pending'}}, {'order_id': 5003, 'date': '2024-02-07', 'total': 75.0, 'items': [{'product': 'Headphones', 'price': 75.0, 'quantity': 1}], 'payment': {'method': 'Credit Card', 'status': 'Completed'}}]}]
Order ID: 5001, Total: 120.5
Order ID: 5002, Total: 250.0
Order ID: 5003, Total: 75.0
  customer_id order_id     product  quantity  total
0         101

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>